# Масштабируемость и обобщение на большее число пользователей

Дополнение к основному ноутбуку [solution.ipynb](solution.ipynb). Основные эксперименты сделаны на сэмпле 100 тыс. пользователей. Здесь два вопроса:

1. **Масштабируемость.** Как растут время расчёта кандидатов, признаков и обучения LightGBM с числом пользователей, вплоть до полного датасета?
2. **Обобщение.** Сохраняются ли выводы основного ноутбука на других размерах выборки: жадный набор сопоставим с ручным, LLM-набор хуже, объединение даёт небольшой прирост?

Наборы признаков заморожены: это наборы, найденные поиском на сэмпле 100 тыс. (`cache/search_*.json`). Поиск заново не запускается. Для каждого размера выполняется тот же код, что в основном ноутбуке: он берётся прямо из `solution.ipynb`, меняется только `N_USERS`. Протокол оценки тот же, что в разделе 6: число деревьев по `train → val`, переобучение на неделе `val`, оценка на тестовой неделе 16–22.08, `seed` LightGBM 42.

Ноутбук рассчитан на Kaggle (Internet: On): там достаточно памяти для полного датасета.

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

ON_KAGGLE = Path('/kaggle/input').exists()
if ON_KAGGLE:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'polars==2.0.0', 'duckdb==2.0.0',
                    'lightgbm==4.7.0', 'openai', 'python-dotenv'], check=True)
    repo = Path('/kaggle/working/recsys-auto-features')
    if not repo.exists():
        subprocess.run(['git', 'clone', '--depth', '1',
                        'https://github.com/404iq/recsys-auto-features.git', str(repo)], check=True)
    os.chdir(repo)
    found = list(Path('/kaggle/input').rglob('interactions.csv'))
    if found:
        os.environ['KION_DATA_DIR'] = str(found[0].parent)
    else:
        url = 'https://github.com/irsafilo/KION_DATASET/raw/f69775be31fa5779907cf0a92ddedb70037fb5ae/data_original.zip'
        subprocess.run(['wget', '-q', url, '-O', 'kion.zip'], check=True)
        subprocess.run(['unzip', '-q', '-j', '-o', 'kion.zip', 'data_original/*.csv', '-d', 'data'], check=True)

Код основного ноутбука. Берём ячейки, которые определяют данные, сэмпл, разбиение, кандидатов, движок признаков, ручной набор и LightGBM. Пропускаем EDA, графики, тест на утечки, перебор `num_leaves` и сами поиски: `num_leaves` фиксируем равным значению, выбранному в основном ноутбуке.

In [ ]:
import gc
import json
import time

try:
    import resource                  # пиковая память процесса; модуль есть только на Linux (Kaggle)
except ImportError:
    resource = None

import numpy as np

SIZES = [10_000, 30_000, 100_000, 300_000, 962_179]   # последний — все пользователи датасета
NUM_LEAVES = 31                                       # выбрано в разделе 3.5 основного ноутбука
if not ON_KAGGLE:
    SIZES = [int(x) for x in os.environ.get('SCAL_SIZES', '5000,10000').split(',')]

nb = json.load(open('solution.ipynb', encoding='utf-8'))
code_cells = [''.join(c['source']) for c in nb['cells'] if c['cell_type'] == 'code']


def pick(first_line=None, contains=None):
    if contains:
        hits = [c for c in code_cells if contains in c]
    else:
        hits = [c for c in code_cells if c.strip().split('\n')[0].startswith(first_line)]
    assert len(hits) == 1, (first_line or contains, len(hits))
    return hits[0]


PIPELINE = [
    pick(contains="MIMALLOC_PURGE_DELAY"),                               # импорты и константы
    pick('interactions = ('),
    pick('rng = np.random.default_rng(SEED)'),
    pick('del interactions'),
    pick('N_POP, N_KNN, KNN_NEIGHBORS, KNN_HALF_LIFE'),
    pick('candidates = {}'),
    pick('IDCG = np.cumsum'),
    pick('import duckdb'),
    pick('@dataclass(frozen=True)'),
    pick('NEG_PER_USER = 60'),
    pick("TIMES = ['all'").split('r = np.random.default_rng(SEED)')[0],   # пространство без теста на утечки
    pick('AGE_ORD = '),
    pick('MANUAL = ['),
    pick('import lightgbm as lgb').split('rows = []\nfor leaves in')[0],  # определения без перебора num_leaves
]
print(f'ячеек основного ноутбука в пайплайне: {len(PIPELINE)}')

Для каждого размера: пайплайн в чистом пространстве имён, затем четыре набора признаков по протоколу раздела 6. Замеряем время этапов и память процесса. Результаты пишутся в `scalability_results.json` после каждого размера, поэтому сбой на полном датасете не теряет уже посчитанное.

In [ ]:
import psutil


def spec_names(path):
    return json.loads(Path(path).read_text(encoding='utf-8'))['accepted']


GREEDY_NAMES = spec_names('cache/search_greedy_random.json')
LLM_NAMES = spec_names('cache/search_greedy_llm.json')


def paired_ci(d, n_boot=2000, seed=42):
    r = np.random.default_rng(seed)
    means = d[r.integers(0, len(d), (n_boot, len(d)))].mean(axis=1)
    return float(d.mean()), *map(float, np.percentile(means, [2.5, 97.5]))


def run_size(n):
    g = {'__name__': '__scal__', 'ON_KAGGLE': ON_KAGGLE}
    t = {}
    for i, src in enumerate(PIPELINE):
        src = src.replace('N_USERS = 100_000', f'N_USERS = {n}')
        exec(compile(src, f'pipeline_{i}', 'exec'), g)
    g['LGB_PARAMS']['num_leaves'] = NUM_LEAVES
    for k in ('X_manual', 'XM'):                         # копии матриц из ячеек основного ноутбука не нужны
        g.pop(k, None)
    gc.collect()
    t['кандидаты (3 недели)'] = sum(v for k, v in g['TIMINGS'].items() if k.startswith('кандидаты '))
    t['ручные признаки, train + val'] = g['TIMINGS'].get('ручные признаки, train + val', float('nan'))

    Spec, cand, XB = g['Spec'], g['cand'], g['XB']

    def to_spec(name):
        key, *rest = name.split('__')
        for prefix in ('ua_', 'di_'):
            if key.startswith(prefix):
                key = prefix[:2] + ':' + key[3:]
        return Spec(key, *rest)

    greedy = [to_spec(x) for x in GREEDY_NAMES]
    llm = [to_spec(x) for x in LLM_NAMES]
    sets = {'только базовые': [], 'ручные': g['MANUAL'], 'жадный, случайный пул': greedy,
            'жадный, пул от LLM': llm, 'ручные + жадный (LLM)': g['MANUAL'] + [s for s in llm if s not in g['MANUAL']]}

    g['setup_fold']('valtr', g['FOLDS']['val'])
    cand['valtr'] = g['make_cand_table']('valtr', g['sample_train'](g['candidates']['val']))
    for f in ['valtr', 'test']:
        XB[f] = g['base_matrix'](f)

    def matrices(specs, folds):
        return {f: np.hstack([XB[f], g['compute_features'](specs, f)]) if specs else XB[f] for f in folds}

    tf = time.perf_counter()
    all_specs = list(dict.fromkeys(g['MANUAL'] + greedy + llm))
    for f in ['train', 'val', 'valtr', 'test']:
        g['compute_features'](all_specs, f)
    t[f'все признаки ({len(all_specs)} шт.), 4 фолда'] = time.perf_counter() - tf

    per_user, fit_times = {}, {}
    for name, specs in sets.items():
        names = g['BASE'] + [s.name for s in specs]
        X = matrices(specs, ('train', 'val'))
        t1 = time.perf_counter()
        model = g['fit_ranker'](X['train'], X['val'], names)
        fit_times[name] = time.perf_counter() - t1
        del X
        Xf = matrices(specs, ('valtr', 'test'))
        dtr = g['lgb'].Dataset(Xf['valtr'], cand['valtr']['target'].to_numpy(), group=g['groups']('valtr'), feature_name=names)
        m = g['lgb'].train(g['LGB_PARAMS'], dtr, model.best_iteration)
        _, pu = g['evaluate'](m.predict(Xf['test']), 'test', per_user=True)
        per_user[name] = pu.sort('user_id')['ndcg'].to_numpy()
        del Xf, dtr, m, model
        gc.collect()
    t['обучение LightGBM, ручной набор'] = fit_times['ручные']

    ref = per_user['ручные']
    res = {
        'пользователей': n,
        'событий в сэмпле': g['log'].height,
        'строк кандидатов val': cand['val'].height,
        'строк обучения train': cand['train'].height,
        'пользователей в тесте': len(ref),
        'память процесса, ГБ': psutil.Process().memory_info().rss / 1e9,
        'пик памяти, ГБ': resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1e6 if resource else float('nan'),
        'время, с': t,
        'ndcg': {k: float(v.mean()) for k, v in per_user.items()},
        'delta': {k: paired_ci(v - ref) for k, v in per_user.items() if k != 'ручные'},
    }
    g['con'].close()
    del g
    gc.collect()
    return res


RESULTS_PATH = Path('scalability_results.json')
results = []
for n in SIZES:
    t0 = time.perf_counter()
    try:
        r = run_size(n)
        r['всего, с'] = time.perf_counter() - t0
        results.append(r)
        print(f"{n:>9,} пользователей: {r['всего, с'] / 60:.1f} мин, NDCG ручных {r['ndcg']['ручные']:.4f}", flush=True)
    except Exception as e:                       # не теряем уже посчитанные размеры
        print(f'{n:,} пользователей: сбой — {type(e).__name__}: {str(e)[:200]}', flush=True)
        gc.collect()
    RESULTS_PATH.write_text(json.dumps(results, ensure_ascii=False, indent=1), encoding='utf-8')

### Время и память

In [ ]:
import polars as pl
import matplotlib.pyplot as plt

COLORS = ['#2a78d6', '#eb6834', '#1baf7a', '#eda100']
plt.rcParams.update({
    'figure.facecolor': '#fcfcfb', 'axes.facecolor': '#fcfcfb', 'axes.edgecolor': '#c3c2b7',
    'axes.labelcolor': '#52514e', 'xtick.color': '#52514e', 'ytick.color': '#52514e',
    'axes.grid': True, 'grid.color': '#e6e5e0', 'grid.linewidth': 0.6,
    'axes.spines.top': False, 'axes.spines.right': False, 'figure.dpi': 110,
})

time_rows = []
for r in results:
    row = {'пользователей': r['пользователей'], 'событий': r['событий в сэмпле'],
           'строк кандидатов val': r['строк кандидатов val']}
    row.update({k: v / 60 for k, v in r['время, с'].items()})
    row['всего, мин'] = r['всего, с'] / 60
    row['память, ГБ'] = r['память процесса, ГБ']
    row['пик памяти, ГБ'] = r['пик памяти, ГБ']
    time_rows.append(row)
time_table = pl.DataFrame(time_rows)
print('время этапов — в минутах')
time_table

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
users = [r['пользователей'] for r in results]
stages = list(results[0]['время, с'])
for stage, color in zip(stages, COLORS):
    ax.plot(users, [r['время, с'][stage] / 60 for r in results], marker='o', linewidth=2, markersize=5, color=color, label=stage)
ax.set_xscale('log')
ax.set_yscale('log')
ax.set_xlabel('пользователей в сэмпле')
ax.set_ylabel('минут')
ax.set_title('Время этапов в зависимости от размера сэмпла', loc='left')
ax.legend(frameon=False, fontsize=8)
fig.tight_layout()
plt.show()

### Сохраняются ли выводы

In [ ]:
gen_rows = []
for r in results:
    row = {'пользователей': r['пользователей'], 'в тесте': r['пользователей в тесте'],
           'NDCG ручных': r['ndcg']['ручные']}
    for name, (d, lo, hi) in r['delta'].items():
        row[f'Δ {name}'] = f'{d:+.4f} [{lo:+.4f}, {hi:+.4f}]'
    gen_rows.append(row)
pl.DataFrame(gen_rows)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
names_g = ['жадный, случайный пул', 'жадный, пул от LLM', 'ручные + жадный (LLM)']
for k, (name, color) in enumerate(zip(names_g, COLORS)):
    x = np.array(users) * (1 + 0.06 * (k - 1))         # небольшой сдвиг, чтобы интервалы не перекрывались
    d = np.array([r['delta'][name] for r in results])
    ax.errorbar(x, d[:, 0], yerr=[d[:, 0] - d[:, 1], d[:, 2] - d[:, 0]], color=color, marker='o',
                markersize=5, linewidth=2, capsize=3, label=name)
ax.axhline(0, color='#52514e', linewidth=1)
ax.set_xscale('log')
ax.set_xlabel('пользователей в сэмпле')
ax.set_ylabel('Δ NDCG@10 к ручному набору')
ax.set_title('Разность с ручным набором на тестовой неделе, 95% ДИ', loc='left')
ax.legend(frameon=False, fontsize=8)
fig.tight_layout()
plt.show()

SCALING_CONCLUSION